# Offshore Update — SL x City mapping

In [ ]:
import re
import glob
import os
import pandas as pd
from openpyxl import load_workbook

## Config — edit paths here if needed

In [ ]:
BASE_DIR = "/Users/ko20689900/Documents/Exersice"
BENCH_DATA_PATH = os.path.join(BASE_DIR, "BenchData.xlsx")
OFFSHORE_DIR = os.path.join(BASE_DIR, "offshore")
SHEET_NAME = "Sheet1"

COUNT_HEADER = "Count"
EMP_HEADER = "Employee Numbers"

## Core functions

In [ ]:
def load_bench_data(path):
    df = pd.read_excel(path, sheet_name=SHEET_NAME)
    df["_SKILL_NORM"] = df["SR_MANDATORY_SKILL"].astype(str).str.strip()
    df["_CITY_NORM"] = df["DERIVED_EMP_CITY"].astype(str).str.strip().str.upper()
    df["_SL_NORM"] = df["SERVICE_LINE"].astype(str).str.strip().str.upper()
    df["_ONOFF_NORM"] = df["ONSITE_OFFSHORE"].astype(str).str.strip().str.upper()
    return df


def get_sl_from_filename(filename):
    match = re.search(r"(SL\d+)", filename, re.IGNORECASE)
    if not match:
        raise ValueError(f"Could not detect SL number from filename: {filename}")
    return match.group(1).upper()


def process_offshore_file(filepath, bench_df):
    sl_value = get_sl_from_filename(os.path.basename(filepath))

    subset = bench_df[
        (bench_df["_SL_NORM"] == sl_value) & (bench_df["_ONOFF_NORM"] == "OFFSHORE")
    ]

    wb = load_workbook(filepath)
    ws = wb[SHEET_NAME] if SHEET_NAME in wb.sheetnames else wb.active

    header_row = 1
    existing_headers = {cell.value: cell.column for cell in ws[header_row] if cell.value}

    if COUNT_HEADER not in existing_headers:
        count_col = ws.max_column + 1
        ws.cell(row=header_row, column=count_col, value=COUNT_HEADER)
    else:
        count_col = existing_headers[COUNT_HEADER]

    if EMP_HEADER not in existing_headers:
        emp_col = ws.max_column + 1
        ws.cell(row=header_row, column=emp_col, value=EMP_HEADER)
    else:
        emp_col = existing_headers[EMP_HEADER]

    last_skill = None
    for row in range(header_row + 1, ws.max_row + 1):
        skill_cell = ws.cell(row=row, column=1).value
        city_cell = ws.cell(row=row, column=2).value

        if skill_cell is not None and str(skill_cell).strip() != "":
            last_skill = str(skill_cell).strip()

        if city_cell is None or str(city_cell).strip() == "":
            continue

        skill = last_skill
        city = str(city_cell).strip().upper()

        matched = subset[
            (subset["_SKILL_NORM"] == skill) & (subset["_CITY_NORM"] == city)
        ]

        count = len(matched)
        emp_numbers = ",".join(matched["EMPNO"].astype(str).tolist())

        ws.cell(row=row, column=count_col, value=count)
        ws.cell(row=row, column=emp_col, value=emp_numbers if count > 0 else "")

    wb.save(filepath)
    print(f"[OFFSHORE] {os.path.basename(filepath)} -> SL={sl_value}, bench rows available: {len(subset)}")

## Run

In [ ]:
bench_df = load_bench_data(BENCH_DATA_PATH)
files = glob.glob(os.path.join(OFFSHORE_DIR, "SL*_OFFSHORE_CITY_Final_Predictions.xlsx"))

if not files:
    print("No offshore files found. Check OFFSHORE_DIR path.")
else:
    for f in sorted(files):
        process_offshore_file(f, bench_df)
    print("Offshore processing complete.")